# Simple Bayesian logistic regression with PyMC

1. Synthesize patients: age, gender, main complaint, and dead/alive within 30 days.
2. Fit a Bayesian logistic regression with N(0, 1) priors.
3. Check that the posterior recovers the true values the data was generated from.

In [ ]:
import numpy as np
import pandas as pd
import pymc as pm

SEED = 42


## 1. Synthesize data

- **age**: Normal(55, 18), clipped to 18–100
- **gender**: Bernoulli(0.5), 1 = male
- **complaint**: one of four, equally likely
- **dead**: Bernoulli(p), where p comes from a logistic model with the true values below.
  The intercept is set so that about 3% die.

Chest pain is the reference complaint, so the other complaints are measured relative to it.

In [47]:
rng = np.random.default_rng(SEED)
N = 5000
COMPLAINTS = ["chest pain", "shortness of breath", "headache", "abdominal pain"]

df = pd.DataFrame({
    "age": rng.normal(55, 18, N).clip(18, 100).round(),
    "male": rng.binomial(1, 0.5, N),
    "complaint": rng.choice(COMPLAINTS, N),
})
df.head()

,age,male,complaint
0,60.0,0,headache
1,36.0,0,chest pain
2,69.0,1,shortness of breath
3,72.0,1,chest pain
4,20.0,0,chest pain


Turn the patients into model inputs. Age is standardised (mean 0, sd 1) so that an
N(0, 1) prior is reasonable for its coefficient. Each complaint except the reference gets a 0/1 column.

In [48]:
X = pd.DataFrame({
    "age": (df["age"] - df["age"].mean()) / df["age"].std(),
    "male": df["male"],
})
for c in COMPLAINTS[1:]:
    X[c] = (df["complaint"] == c).astype(int)
X.head()

,age,male,shortness of breath,headache,abdominal pain
0,0.299643,0,0,1,0
1,-1.067861,0,0,0,0
2,0.812457,1,1,0,0
3,0.983394,1,0,0,0
4,-1.979529,0,0,0,0


Set the true betas and intercept, then compute the log-odds of death for each patient and convert them to a true probability of death with the sigmoid. Finally, draw each patient's outcome (dead/alive) from a Bernoulli distribution with that probability.

In [55]:
# the true values PyMC should recover
rng = np.random.default_rng(SEED)
TRUE_INTERCEPT = -3.9
TRUE_BETA = pd.Series({
    "age": 0.8,                   # per standard deviation of age (about 18 years)
    "male": 0.3,
    "shortness of breath": 0.5,
    "headache": -0.7,
    "abdominal pain": -0.2,
})

p_true = 1 / (1 + np.exp(-(TRUE_INTERCEPT + X @ TRUE_BETA)))
df["dead"] = rng.binomial(1, p_true)
print(f"{df['dead'].sum()} deaths of {N} ({df['dead'].mean():.1%})")

167 deaths of 5000 (3.3%)


## 2. The model

    intercept ~ N(0, 1)
    beta_j    ~ N(0, 1)            one per column of X
    p_i       = sigmoid(intercept + x_i · beta)
    dead_i    ~ Bernoulli(p_i)

`pm.sample` draws from the posterior with NUTS (4 chains × 1000 draws).

Prior: \
log ods for the intecept is a normal distribuion around log ods 0 with, deviation 1 \
beta coefficients normaly distributed around 0, standard deviation 1 



In [52]:
with pm.Model(coords={"feature": X.columns}) as model:
    intercept = pm.Normal("intercept", mu=0, sigma=1)
    beta = pm.Normal("beta", mu=0, sigma=1, dims="feature")

    p = pm.math.sigmoid(intercept + pm.math.dot(X.values, beta))
    pm.Bernoulli("dead", p=p, observed=df["dead"].values)

    idata = pm.sample(1000, tune=1000, chains=4, random_seed=SEED)

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [intercept, beta]


/home/theo/Work/DEV/skola/pda/.venv/lib/python3.14/site-packages/rich/live.py:260: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 12 seconds.


## 3. Posterior vs truth

Each parameter's posterior is 4000 draws. The mean is the best estimate and the
2.5%–97.5% range is the 95% credible interval. The truth should usually fall inside it.

In [6]:
draws = {"intercept": idata.posterior["intercept"].values.ravel()}
beta_draws = idata.posterior["beta"].values.reshape(-1, X.shape[1])
for j, name in enumerate(X.columns):
    draws[name] = beta_draws[:, j]

truth = {"intercept": TRUE_INTERCEPT, **TRUE_BETA}
result = pd.DataFrame({
    "true": truth,
    "mean": {k: v.mean() for k, v in draws.items()},
    "2.5%": {k: np.quantile(v, 0.025) for k, v in draws.items()},
    "97.5%": {k: np.quantile(v, 0.975) for k, v in draws.items()},
})
result["inside"] = (result["2.5%"] <= result["true"]) & (result["true"] <= result["97.5%"])
result.round(2)

,true,mean,2.5%,97.5%,inside
intercept,-3.9,-3.50,-3.84,-3.17,False
age,0.8,0.68,0.51,0.84,True
male,0.3,0.15,-0.17,0.50,True
shortness of breath,0.5,0.25,-0.12,0.63,True
headache,-0.7,-1.23,-1.79,-0.71,False
abdominal pain,-0.2,-0.74,-1.22,-0.27,False


## 4. Predict for one patient

A 70-year-old man with shortness of breath. Every posterior draw gives one probability,
so the prediction is a distribution, not a single number.

In [59]:
patient = pd.Series(0.0, index=X.columns)
patient["age"] = (70 - df["age"].mean()) / df["age"].std()
patient["male"] = 1
patient["shortness of breath"] = 1

logit_draws = draws["intercept"] + beta_draws @ patient.values
p_draws = 1 / (1 + np.exp(-logit_draws))
p_true_patient = 1 / (1 + np.exp(-(TRUE_INTERCEPT + patient @ TRUE_BETA)))

print(f"true P(death):      {p_true_patient:.3f}")
print(f"posterior mean:     {p_draws.mean():.3f}")
print(f"95% interval:       {np.quantile(p_draws, 0.025):.3f} – {np.quantile(p_draws, 0.975):.3f}")

true P(death):      0.083
posterior mean:     0.077
95% interval:       0.057 – 0.099
